In [0]:
# replace with your catalog
CATALOG = spark.catalog.currentCatalog()
CATALOG = "nikkthegreek"

In [0]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import DataFrame
import json

In [0]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

In [0]:
try:
    spark
except NameError:
    builder = (
        SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
        .master("local[4]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 1. Set Up and Data

In [0]:
res = []
resource = "planets"
query = f"https://swapi.tech/api/{resource}"
json_request = requests.get(query).json()
res.extend(json_request["results"])

while json_request["next"]:
    json_request = requests.get(json_request["next"]).json()
    res.extend(json_request["results"])

In [0]:
res

In [0]:
df = spark.createDataFrame(res)
df.show()

In [0]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [0]:
df = df.withColumn("properties", get_properties(F.col("url")))

In [0]:
df.show()

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

In [0]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Overwrite

In [0]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [0]:
class StarWarsBronze2(bronze.Bronze):
    def custom_load(self, table):
        df = spark.range(5).withColumn("t", F.lit(table))
        return df


bronze_instance2 = StarWarsBronze2(spark, **options)

In [0]:
# Ensure the schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

# Run one table
bronze_instance.load().transform().write(mode="overwrite").execute("people")

In [0]:
# run multiple tables
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show(100)

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(100)

In [0]:
bronze_instance.data["people"].show()

In [0]:
bronze_instance.data["planets"].show()

# 3 Replace Where

In [0]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))

    def get_replace_condition(self, df: DataFrame, table: str) -> str:
        return "uid > '0'"


bronze_instance = StarWarsBronze(spark, **options)
bronze_instance.load().transform().write(mode="replace").execute("people")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

# 4 Append

In [0]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)
bronze_instance.load().transform().write().execute("people")  # default mode is append

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

# 5 Merge

In [0]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))

    def get_delta_merge_builder(
        self, df: DataFrame, delta_table: DeltaTable
    ) -> DeltaMergeBuilder:
        merge_condition = "target.uid = source.uid"
        builder = delta_table.alias("target").merge(df.alias("source"), merge_condition)
        builder = builder.whenMatchedUpdateAll()
        builder = builder.whenNotMatchedInsertAll()
        return builder


bronze_instance = StarWarsBronze(spark, **options)
bronze_instance.load().transform().write(mode="merge").execute("people")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

# 6 Clean Up

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
#spark.stop()